<a href="https://colab.research.google.com/github/SubeenJung0228/Problem-Set-econ-3916/blob/main/Problem%20Set%202/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [35]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean = 3.5, sigma = 0.6, size = n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.random(n) < 0.22
    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size = n)
    satisfaction = np.clip(np.rint(s), 1, 5)

    return pd.DataFrame ({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction,

    })

pop_before = simulate_population(BASE_BEFORE)
pop_after = simulate_population(BASE_AFTER)
print(f"true mean before: {pop_before['satisfaction'].mean():.3f}")
print(f"true mean after:  {pop_after['satisfaction'].mean():.3f}")

true mean before: 3.291
true mean after:  3.391


*Your answer here.*

I simulated 50,000 users before and after the redesign. The average satisfaction score went from 3.291 to 3.391, which means satisfaction increased by 0.100 points. In real life, we can’t know the true satisfaction of every user, but since I created the simulated data, I know the actual values. I can use these values as a baseline to compare with the survey results later and see how close the PM’s estimates are to the true improvement.

### Step 1.2: Impose the response mechanism (15 pts)

In [36]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    p = 1 / (1 + np.exp(-(a + b * (df["s"] - 3.3) + 0.30 * df["engagement_z"] - 0.25 * df["had_support_ticket"])))
    return rng.random(len(df)) < p

resp_before = respond(pop_before, a=-1.0, b=0.45)
resp_after = respond(pop_after, a=-2.75, b=2.00)

obs_before = pop_before["satisfaction"][resp_before].mean()
obs_after = pop_after["satisfaction"][resp_after].mean()

print(f"observed mean before: {obs_before:.3f} | response rate {resp_before.mean():.1%}")
print(f"observed mean after:  {obs_after:.3f} | response rate {resp_after.mean():.1%}")
print(f"apparent gap: {obs_after - obs_before:.3f}")

observed mean before: 3.592 | response rate 27.5%
observed mean after:  4.309 | response rate 15.3%
apparent gap: 0.717


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

This is nonresponse bias, one of Chapter 6’s forms of selection bias. Since satisfaction affects whether users respond, Cov(R,Y) ≠ 0, so the missing data are MNAR. After the redesign, b rose from 0.45 to 2.00, and respondents were even more satisfied than the overall population. The observed mean rose from 3.592 to 4.309, a gap of 0.717 points, while the true improvement was only 0.100 points.

A larger sample would not fix this bias. Meng’s identity shows that the error depends on selection bias, satisfaction variability, and the proportion of missing data. More responses through the same mechanism would give a more precise but still biased estimate, like the Literary Digest poll.

The response rate also fell from 27.5% to 15.3%, which suggests that the apparent improvement came mainly from who responded rather than a real increase in satisfaction.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [37]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    return np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])

sample_after = pop_after["satisfaction"][resp_after].values

# mean, by hand
boot_means = bootstrap_by_hand(sample_after, np.mean)
lo, hi = np.percentile(boot_means, [2.5, 97.5])
print(f"mean, by hand: [{lo:.3f}, {hi:.3f}]")

# median, by hand
boot_medians = bootstrap_by_hand(sample_after, np.median)
lo_med, hi_med = np.percentile(boot_medians, [2.5, 97.5])
print(f"median, by hand: [{lo_med:.3f}, {hi_med:.3f}]")

# scipy: mean
mean_pct = stats.bootstrap((sample_after,), np.mean, n_resamples=2000, method="percentile", random_state=1).confidence_interval
mean_bca = stats.bootstrap((sample_after,), np.mean, n_resamples=2000, method="BCa", random_state=1).confidence_interval
print(f"mean, scipy percentile: [{mean_pct.low:.3f}, {mean_pct.high:.3f}]")
print(f"mean, scipy BCa:        [{mean_bca.low:.3f}, {mean_bca.high:.3f}]")

# scipy: median
med_pct = stats.bootstrap((sample_after,), np.median, n_resamples=2000, method="percentile", random_state=1).confidence_interval
med_bca = stats.bootstrap((sample_after,), np.median, n_resamples=2000, method="BCa", random_state=1).confidence_interval
print(f"median, scipy percentile: [{med_pct.low:.3f}, {med_pct.high:.3f}]")
print(f"median, scipy BCa:        [{med_bca.low:.3f}, {med_bca.high:.3f}]")

mean, by hand: [4.294, 4.325]
median, by hand: [4.000, 4.000]
mean, scipy percentile: [4.293, 4.325]
mean, scipy BCa:        [4.294, 4.325]
median, scipy percentile: [4.000, 4.000]
median, scipy BCa:        [nan, nan]


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


*Your answer here.*

I bootstrapped the mean and median of post-redesign satisfaction using my own loop and **scipy.stats.bootstrap**. Both methods gave the same percentile intervals: [4.294, 4.325] for the mean and [4, 4] for the median.

For the mean, the percentile and BCa intervals were also identical because the bootstrap distribution was nearly symmetric. For the median, BCa returned nan because the satisfaction scores were whole numbers and most respondents chose 4, so every resample had a median of 4.

However, these narrow intervals do not mean the estimates are accurate. The true mean was 3.391, far below the interval. Bootstrap measures sampling uncertainty, not nonresponse bias, so it can give a precise interval around the wrong value.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [38]:
hits, n_reps = 0, 400

for _ in range(n_reps):
    idx = rng.choice(N_USERS, size=2000, replace=False)
    r = respond(pop_after.iloc[idx], a=-2.75, b=2.00, rng=rng).values
    s = pop_after["satisfaction"].values[idx][r]
    se = s.std(ddof=1) / np.sqrt(len(s))
    hits += (s.mean() - 1.96*se) <= pop_after["satisfaction"].mean() <= (s.mean() + 1.96*se)

coverage = hits / n_reps
mc_se = np.sqrt(coverage * (1 - coverage) / n_reps)
print(f'coverage {coverage:.3f} ± {mc_se:.3f} (Monte Carlo SE)')

coverage 0.000 ± 0.000 (Monte Carlo SE)


*Your answer here.*

I repeated the survey 400 times and built a 95% confidence interval for the respondents’ mean. None of the intervals contained the true mean of 3.391, so the coverage was 0%. With zero successes out of 400 trials, the rule of three suggests the true coverage is below about 0.75%.

The problem is not the interval formula but the nonresponse bias. Since happier users were more likely to respond, the estimates stayed near 4.3, far above the true mean. The intervals measured sampling uncertainty but failed to account for bias, so they did not provide the promised 95% coverage.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [39]:
sample_before = pop_before["satisfaction"][resp_before].values

t_stat, p_value = stats.ttest_ind(sample_after, sample_before, equal_var=False)

diff = sample_after.mean() - sample_before.mean()
pooled_sd = np.sqrt((sample_after.var(ddof=1) + sample_before.var(ddof=1)) / 2)
cohens_d = diff / pooled_sd

print(f"observed difference: {diff:.3f} points")
print(f"Welch's t = {t_stat:.2f}, p = {p_value:.3g}")
print(f"Cohen's d = {cohens_d:.3f}")

observed difference: 0.717 points
Welch's t = 66.33, p = 0
Cohen's d = 0.911


*Your answer here.*

The PM’s test shows a significant difference: the post-redesign mean was 0.717 points higher (t= 66.33, p < 0.001), with a large Cohen’s d of 0.911.

However, statistical significance does not mean the results are accurate. The test compares two biased samples and cannot show whether satisfaction improved for all users. Most of the observed gap comes from nonresponse bias, while the true improvement is only 0.100 points. With such a large sample, even a biased difference can be statistically significant.

### Step 3.2: Power, before the fact

In [40]:
from statsmodels.stats.power import TTestIndPower

In [41]:
# Sample size needed
sd = pop_before["satisfaction"].std(ddof=1)
d_true = 0.10 / sd
n_needed = TTestIndPower().solve_power(effect_size=d_true, alpha=0.05, power=0.80)
print(f"d for a 0.10-point improvement: {d_true:.3f}")
print(f"n per group for 80% power: {n_needed:,.0f}")

# Detectable effect
n_resp = resp_after.sum()
d_min = TTestIndPower().solve_power(nobs1=n_resp, alpha=0.05, power=0.80)
print(f"respondents after the redesign: {n_resp:,}")
print(f"smallest detectable d: {d_min:.3f} = {d_min * sd:.3f} points")

# Does a larger sample help?
true_after = pop_after["satisfaction"].mean()
for size in [200, 800, 3200, 12800]:
    errors = []
    for _ in range(200):
        idx = rng.choice(N_USERS, size=size, replace=False)
        r = respond(pop_after.iloc[idx], a=-2.75, b=2.00, rng=rng).values
        s = pop_after["satisfaction"].values[idx][r]
        errors.append(s.mean() - true_after)
    errors = np.array(errors)
    print(f"surveyed {size:>6,}: mean error {errors.mean():+.3f}, sd of error {errors.std():.3f}")


d for a 0.10-point improvement: 0.112
n per group for 80% power: 1,249
respondents after the redesign: 7,638
smallest detectable d: 0.045 = 0.040 points
surveyed    200: mean error +0.914, sd of error 0.110
surveyed    800: mean error +0.924, sd of error 0.060
surveyed  3,200: mean error +0.920, sd of error 0.030
surveyed 12,800: mean error +0.920, sd of error 0.014


*Your answer here.*

(1) **Sample size needed.** A 0.10-point improvement gives an effect size of d=0.112. To detect it with 80% power at alpha = 0.05, we need about 1,249 respondents per group.

(2) **Detectable effect.** With 7,638 respondents, the survey can detect an effect as small as d=0.045, or about 0.040 points, so the sample is large enough to detect the true improvement.

(3) **Does a larger sample help?** I simulated surveys with 200, 800, 3,200, and 12,800 users, repeating each 200 times. The mean error stayed around +0.92 points, while its spread decreased from 0.110 to 0.014. A larger sample made the estimate more precise, but not more accurate.

Power analysis assumes the estimate is centered on the true effect. Here, nonresponse bias causes the error because happier users are more likely to respond. More data cannot fix this bias; we need a better survey design, such as randomly selecting users and following up with non-respondents.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [42]:
# Assumes pop_after (DataFrame) and resp_after (boolean array, same row order)
# already exist in your session. No new data is simulated.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]
TRUE_MEAN = pop_after["satisfaction"].mean()


def ipw_mean(X, y, responded, trim=0.01):
    """Hajek IPW mean: fit P(respond | X) on everyone, weight respondents by 1/p."""
    model = LogisticRegression(C=1e6, max_iter=1000)  # effectively unpenalized
    model.fit(X, responded.astype(int))
    p_hat = np.clip(model.predict_proba(X)[:, 1], trim, 1.0)
    w = 1.0 / p_hat[responded]
    return np.sum(w * y[responded]) / np.sum(w)


resp = np.asarray(resp_after, dtype=bool)
assert len(resp) == len(pop_after), "resp_after must align row-for-row with pop_after"

X = pop_after[COVARIATES].to_numpy(dtype=float)
y = pop_after["satisfaction"].to_numpy(dtype=float)  # only respondents' values are used

naive = y[resp].mean()
ipw_est = ipw_mean(X, y, resp)

# Bootstrap: resample all units (respondents and non-respondents), refit each time
rng = np.random.default_rng(42)
n_boot = 1000
n = len(resp)
boot = np.empty(n_boot)
for b in range(n_boot):
    idx = rng.integers(0, n, n)
    boot[b] = ipw_mean(X[idx], y[idx], resp[idx])
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])

print(f"Response rate        : {resp.mean():.3f}")
print(f"Naive respondent mean: {naive:.3f}  (bias {naive - TRUE_MEAN:+.3f})")
print(f"IPW weighted estimate: {ipw_est:.3f}  (bias {ipw_est - TRUE_MEAN:+.3f})")
print(f"95% bootstrap CI     : [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"True population mean : {TRUE_MEAN:.3f}  "
      f"(inside CI: {ci_lo <= TRUE_MEAN <= ci_hi})")

Response rate        : 0.153
Naive respondent mean: 4.309  (bias +0.918)
IPW weighted estimate: 4.137  (bias +0.746)
95% bootstrap CI     : [4.117, 4.158]
True population mean : 3.391  (inside CI: False)


*Your answer here.*

1. Your revision

Version 1 did not include my actual variable names, so the model created its own data and used the wrong true mean (3.664 instead
of 3.391) and response rate (41.8% instead of 15.3%). In Version 2, I specified my existing data, the separate resp_after array, the random seed, and the true mean to ensure the code used my actual data. I also replaced the hard-coded true mean with pop_after["satisfaction"].mean(), so the check uses my population’s actual true mean of 3.391.

2. Your check

The weighted estimate was 4.137, with a 95% bootstrap interval of [4.117, 4.158], far from the true mean of 3.391. IPW reduced the bias from +0.918 to +0.746 but did not eliminate it because the response model used engagement and support tickets, not satisfaction itself. Since response depended on satisfaction (MNAR), most of the bias remained. Including satisfaction in the model could reduce the bias in the simulation, but this is not practical because non-respondents’ satisfaction is unknown.

3. In your own words

The code uses logistic regression to estimate each user’s response probability based on engagement and support tickets. Each respondent’s weight is the inverse of that probability, so users who are less likely to respond receive more weight. The weighted mean is the sum of weighted satisfaction scores divided by the sum of the weights.

4. One sentence

I would use AI again because it can write the weighting and bootstrap code quickly, but I would provide my actual data and check the results against the known true mean to avoid errors from simulated data.

### Task 4.2: The slide (5 pts)

*Your answer here.*

The redesign survey cannot confirm whether satisfaction improved. Respondents had a high mean satisfaction score of 4.31 (95% bootstrap CI: 4.29–4.33), but the response rate fell from 27.5% to 15.3%. The apparent 0.72-point improvement is mostly due to happier users being more likely to respond, while the true improvement is only 0.10 points. Weighting by engagement and support tickets does not remove the bias, and a larger sample would only make the estimate more precise, not more accurate. To reduce this bias, we should randomly select users and follow up with non-respondents to get a more representative sample.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:*

Opus 5.5 / Oct 23  

*Version 1 prompt:*

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

*Reply:*

Mechanism check. The correction is valid under missing at random (MAR) given the covariates, also called ignorable non-response or conditional independence. Formally, R ⊥ Y | X: once you know engagement_minutes and had_support_ticket, whether someone responds carries no further information about their satisfaction. Equivalently, P(R=1 | Y, X) = P(R=1 | X). Under that assumption, E[R·Y / π(X)] = E[Y]. Two more conditions are needed. Positivity requires π(X) > 0 for every covariate pattern in the population. The logistic model for π(X) must also be correctly specified.

When response depends on satisfaction itself, you have missing not at random (MNAR). The true response probability is π(Y, X), but your model can only estimate P(R=1 | X) = E[π(Y, X) | X], which averages over the satisfaction distribution within each covariate cell. Inside a cell, respondents are still the more satisfied users, so the weighted mean of respondents in that cell overestimates the cell’s true mean. Reweighting the cells correctly does nothing about this within-cell selection. The correction only removes the part of the bias that runs through the covariates, meaning the part of satisfaction they predict. The bias from the residual, the variation in satisfaction the covariates don’t explain, remains. That is why the demo removes only about 27% of the bias. The oracle, which conditions on satisfaction, removes essentially all of it, which shows the failure comes from the missing variable and not from the method.

Evaluate: No. Under your mechanism, the covariate-based IPW will not recover the true mean. Response depends directly on the outcome, so MAR given engagement_minutes and had_support_ticket is violated. The estimator converges to a biased value, here roughly +0.45. More data won’t help, because the bootstrap interval only gets narrower around the wrong value, as the demo’s interval shows by missing the truth. The single exception would be a population in which the covariates fully determine satisfaction, leaving no residual variation; real survey populations don’t look like that.

The data alone can’t fix this. Realistic options include:

Finding auxiliary variables strongly predictive of satisfaction, such as churn, NPS proxies, or usage drop-off, to make MAR more plausible.
Following up a random subsample of non-respondents intensively, called double sampling.
Using instruments that affect response but not satisfaction, such as randomized incentives or contact timing, in a selection model.
Reporting a sensitivity analysis, for example a pattern-mixture or δ-adjustment that shifts non-respondent means by an assumed amount.



```
"""
Inverse-probability weighting (IPW) for survey non-response.

Expects a DataFrame with one row per sampled user:
  satisfaction        observed only where responded == 1
  engagement_minutes  observed for everyone (frame / telemetry data)
  had_support_ticket  observed for everyone (0/1)
  responded           0/1 response indicator
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]


def fit_response_propensity(df, covariates=COVARIATES):
    """Logistic model P(responded = 1 | covariates), fit on the full sample."""
    X = df[covariates].to_numpy(dtype=float)
    y = df["responded"].to_numpy()
    model = LogisticRegression(C=1e6, max_iter=1000)  # effectively unpenalized
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]


def ipw_mean(df, covariates=COVARIATES, trim=0.01):
    """Hajek (normalized) IPW estimate of mean satisfaction."""
    p_hat = np.clip(fit_response_propensity(df, covariates), trim, 1.0)
    r = df["responded"].to_numpy() == 1
    w = 1.0 / p_hat[r]
    y = df.loc[r, "satisfaction"].to_numpy()
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, covariates=COVARIATES, n_boot=1000, alpha=0.05, seed=0):
    """Nonparametric bootstrap: resample the FULL sample (respondents and
    non-respondents) and refit the propensity model each time, so the
    interval reflects uncertainty in the estimated weights."""
    rng = np.random.default_rng(seed)
    n = len(df)
    est = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        est[b] = ipw_mean(df.iloc[idx], covariates)
    lo, hi = np.quantile(est, [alpha / 2, 1 - alpha / 2])
    return ipw_mean(df, covariates), (lo, hi)


# ---------------------------------------------------------------------------
# Demo on a simulated population (replace with your own df)
# ---------------------------------------------------------------------------
def simulate(n=20_000, seed=42):
    rng = np.random.default_rng(seed)
    engagement_minutes = rng.gamma(shape=2.0, scale=30.0, size=n)
    had_support_ticket = rng.binomial(1, 1 / (1 + np.exp(-(-1.5 + 0.01 * engagement_minutes))))
    satisfaction = (3.0 + 0.015 * engagement_minutes - 0.8 * had_support_ticket
                    + rng.normal(0, 1.0, n))
    # Non-response depends on satisfaction ITSELF (MNAR)
    p_respond = 1 / (1 + np.exp(-(-1.0 + 0.9 * (satisfaction - 3.0))))
    responded = rng.binomial(1, p_respond)
    return pd.DataFrame(dict(satisfaction=satisfaction,
                             engagement_minutes=engagement_minutes,
                             had_support_ticket=had_support_ticket,
                             responded=responded))


if __name__ == "__main__":
    df = simulate()
    true_mean = df["satisfaction"].mean()              # known only in simulation
    naive = df.loc[df.responded == 1, "satisfaction"].mean()
    est, (lo, hi) = bootstrap_ipw(df, n_boot=500)

    # Oracle: propensity model that includes satisfaction (impossible in practice)
    oracle = ipw_mean(df, COVARIATES + ["satisfaction"])

    print(f"Response rate          : {df.responded.mean():.3f}")
    print(f"True population mean   : {true_mean:.3f}")
    print(f"Naive respondent mean  : {naive:.3f}  (bias {naive - true_mean:+.3f})")
    print(f"IPW on covariates      : {est:.3f}  (bias {est - true_mean:+.3f})")
    print(f"  95% bootstrap CI     : [{lo:.3f}, {hi:.3f}]  "
          f"covers truth: {lo <= true_mean <= hi}")
    print(f"Oracle IPW (+satisf.)  : {oracle:.3f}  (bias {oracle - true_mean:+.3f})")

```



*Version 2 prompt:*

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.Use the existing pop_after DataFrame and the resp_after array without simulating new data. resp_after is a separate boolean array that marks who responded; it is not a column of pop_after. Calculate the weighted estimate, bootstrap its interval using rng = np.random.default_rng(42), and print the weighted estimate next to the true population mean of 3.394 for comparison.  

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.


**Change**: Version 1 did not specify my variable names, so the model created its own data. Version 2 uses my existing pop_after and separate resp_after array, my random seed, and the true mean.

*Reply:*

Mechanism check. The correction is valid under missing at random given the covariates: R ⊥ satisfaction | (engagement_minutes, had_support_ticket). In words, once you know the two covariates, whether someone responds says nothing more about their satisfaction. Two further conditions are needed: every covariate pattern must have a nonzero chance of responding (positivity), and the logistic model must be correctly specified.

If response depends on satisfaction itself, that assumption breaks. Within any covariate cell, the people who respond are still the more satisfied ones. Reweighting the cells corrects for differences between cells, but not for this selection inside them. So the estimator removes only the part of the bias that the covariates predict. The bias from satisfaction the covariates don’t explain remains.

Evaluate: No. Your mechanism ties response directly to satisfaction, so the weighted estimate will generally stay above 3.394. It should land between the naive respondent mean and the truth. The bootstrap interval measures sampling noise only, not this bias, so it will likely be narrow and miss 3.394. The only exception is if your two covariates explain nearly all the variation in satisfaction. To do better you’d need stronger predictors of satisfaction, a follow-up of non-respondents, or a sensitivity analysis for MNAR (missing not at random).



```
# Assumes pop_after (DataFrame) and resp_after (boolean array, same row order)
# already exist in your session. No new data is simulated.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]
TRUE_MEAN = 3.394


def ipw_mean(X, y, responded, trim=0.01):
    """Hajek IPW mean: fit P(respond | X) on everyone, weight respondents by 1/p."""
    model = LogisticRegression(C=1e6, max_iter=1000)  # effectively unpenalized
    model.fit(X, responded.astype(int))
    p_hat = np.clip(model.predict_proba(X)[:, 1], trim, 1.0)
    w = 1.0 / p_hat[responded]
    return np.sum(w * y[responded]) / np.sum(w)


resp = np.asarray(resp_after, dtype=bool)
assert len(resp) == len(pop_after), "resp_after must align row-for-row with pop_after"

X = pop_after[COVARIATES].to_numpy(dtype=float)
y = pop_after["satisfaction"].to_numpy(dtype=float)  # only respondents' values are used

naive = y[resp].mean()
ipw_est = ipw_mean(X, y, resp)

# Bootstrap: resample all units (respondents and non-respondents), refit each time
rng = np.random.default_rng(42)
n_boot = 1000
n = len(resp)
boot = np.empty(n_boot)
for b in range(n_boot):
    idx = rng.integers(0, n, n)
    boot[b] = ipw_mean(X[idx], y[idx], resp[idx])
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])

print(f"Response rate        : {resp.mean():.3f}")
print(f"Naive respondent mean: {naive:.3f}  (bias {naive - TRUE_MEAN:+.3f})")
print(f"IPW weighted estimate: {ipw_est:.3f}  (bias {ipw_est - TRUE_MEAN:+.3f})")
print(f"95% bootstrap CI     : [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"True population mean : {TRUE_MEAN:.3f}  "
      f"(inside CI: {ci_lo <= TRUE_MEAN <= ci_hi})")

```

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.